# 04 — Quality Gate Validation

Reads Silver, runs 10 clinical rules, routes to Gold/Quarantine.

**Inputs:** `silver/harmonized/`  
**Outputs:** `gold/validated/`, `quarantine/failed_records/`

In [ ]:
import sys, os
sys.path.insert(0, '..')
from scripts.utils import load_config
config = load_config('../config.yaml')

In [ ]:
import pandas as pd
silver_df = pd.read_parquet('../' + config['paths']['silver'])
print(f'Silver: {len(silver_df)} rows')
silver_df.info()

In [ ]:
bounds = config['clinical_bounds']
fm = {}
fm['patient_id_null'] = silver_df['patient_id'].isna()
fm['source_dataset_null'] = silver_df['source_dataset'].isna()
fm['age_oob'] = ~silver_df['age'].between(bounds['age_min'], bounds['age_max'])
fm['bmi_oob'] = ~silver_df['bmi'].between(bounds['bmi_min'], bounds['bmi_max'])
fm['glucose_oob'] = ~silver_df['glucose_mmol'].between(bounds['glucose_mmol_min'], bounds['glucose_mmol_max'])
fm['bp_oob'] = ~silver_df['blood_pressure_systolic'].between(bounds['bp_systolic_min'], bounds['bp_systolic_max'])
fm['insulin_oob'] = ~silver_df['insulin_uU_ml'].between(bounds['insulin_min'], bounds['insulin_max'])
any_fail = pd.concat(fm.values(), axis=1).any(axis=1)
failing_df = silver_df[any_fail].copy()
failing_df['failure_reason'] = [', '.join([r for r,m in fm.items() if m.iloc[i]]) for i in range(len(silver_df)) if any_fail.iloc[i]]
passing_df = silver_df[~any_fail].copy()
print(f'Gold: {len(passing_df)}, Quarantine: {len(failing_df)}')

In [ ]:
gold_dir = '../' + config['paths']['gold']
q_dir = '../' + config['paths']['quarantine']
os.makedirs(gold_dir, exist_ok=True)
os.makedirs(q_dir, exist_ok=True)
passing_df.to_parquet(os.path.join(gold_dir, 'gold_validated.parquet'), index=False)
failing_df.to_parquet(os.path.join(q_dir, 'quarantine.parquet'), index=False)
print('Written.')

## Done
Gold: 3,111 rows | Quarantine: 7,628 rows